# unsqueeze, squeeze et None : ajouter un axe de canal

Ajouter une dimension de taille un sans fabriquer de nouvelles mesures.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#ajouter-retirer-un-axe).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Une couche audio peut attendre (extraits,canaux,temps). Notre signal possède seulement (extraits,temps), soit (1,3). Nous allons ajouter l’axe de canal manquant.

| Expression | Axes | Forme |
| --- | --- | --- |
| signal | extraits,temps | (1,3) |
| signal.unsqueeze(1) | extraits,canaux,temps | (1,1,3) |
| avec_canal.squeeze(1) | extraits,temps | (1,3) |
| avec_canal.squeeze() | temps seulement | (3,) |

Les valeurs 1,2,3 restent les mêmes. Le danger vient des axes retirés, pas d’un changement de nombres.

## Démonstration 1 · Insérer un canal puis comparer deux retraits

Choisir explicitement la position que nous voulons modifier.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

Les crochets extérieurs gardent un axe d’extraits de taille un.

In [4]:
signal = torch.tensor([[1., 2., 3.]])


In [5]:
observer('signal', signal)

signal forme (1, 3) type torch.float32 requires_grad False
  valeurs : [[1.0, 2.0, 3.0]]


### Calculer l’état suivant

L’axe 0 existe déjà. unsqueeze(1) ajoute le nouveau canal entre l’extrait et le temps.

In [6]:
avec_canal = signal.unsqueeze(1)


In [7]:
observer('avec_canal', avec_canal)

avec_canal forme (1, 1, 3) type torch.float32 requires_grad False
  valeurs : [[[1.0, 2.0, 3.0]]]


### Calculer l’état suivant

squeeze(1) retire l’axe un si sa taille vaut un ; il conserve l’axe des extraits.

In [8]:
retour = avec_canal.squeeze(1)


In [9]:
observer('retour', retour)

retour forme (1, 3) type torch.float32 requires_grad False
  valeurs : [[1.0, 2.0, 3.0]]


### Calculer l’état suivant

squeeze() sans argument retire tous les axes unitaires, y compris l’axe des extraits.

In [10]:
tout_retire = avec_canal.squeeze()


In [11]:
observer('tout_retire', tout_retire)

tout_retire forme (3,) type torch.float32 requires_grad False
  valeurs : [1.0, 2.0, 3.0]


### Lire les sorties

signal[:,None,:] garde tous les extraits, insère un axe, puis garde tout le temps. None ne signifie pas une valeur manquante ici.

In [12]:
print("signal", signal.tolist(), tuple(signal.shape))
print("canal", avec_canal.tolist(), tuple(avec_canal.shape))
print("squeeze(1)", retour.tolist(), tuple(retour.shape))
print("squeeze()", tout_retire.tolist(), tuple(tout_retire.shape))


signal [[1.0, 2.0, 3.0]] (1, 3)
canal [[[1.0, 2.0, 3.0]]] (1, 1, 3)
squeeze(1) [[1.0, 2.0, 3.0]] (1, 3)
squeeze() [1.0, 2.0, 3.0] (3,)


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [13]:
print("None equivalent", torch.equal(signal[:, None, :], avec_canal))

None equivalent True


L’axe de canal est une organisation du même signal. Nous devons conserver l’axe batch même lorsque le lot ne contient qu’un extrait.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Ajoutez l’axe canal en position 1 à une entrée (3,2). Prévoyez la forme.

In [14]:
exo_x = torch.zeros(3,2)
exo_canal = exo_x.unsqueeze(1)
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(exo_canal.shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Les valeurs ne sont ni répétées ni mélangées. Un axe de taille un est inséré.

</details>

### Exercice 2

Retirez uniquement cet axe canal de taille un. Quelle forme reste-t-il ?

In [15]:
exo_x = torch.zeros(3,1,2)
exo_sans_canal = exo_x.squeeze(1)
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(exo_sans_canal.shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

squeeze(1) vise un axe nommé ; il ne retire pas les autres axes.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.zeros(3,2)
exo_canal = exo_x.unsqueeze(1)
exo_forme = (3,1,2)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(exo_canal.shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.zeros(3,1,2)
exo_sans_canal = exo_x.squeeze(1)
exo_forme = (3,2)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(exo_sans_canal.shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [16]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()